# Task 2 — Test 18종 보강 (원본 + 합성 pool2, 74종 학습)

**목적**: test에만 존재하는 추가 18종 + train 희소 클래스를 합성 이미지(pool2)로 균형 배치하여 74종 전체를 학습한다.

| 항목 | 내용 |
|---|---|
| 학습 데이터 | 원본 train 232장 + 합성 pool2 (test 전용 18종 + train 희소 클래스 균형 배치) |
| 클래스 | 74종 (train 56종 → 라벨 1~56, test 전용 18종 → 라벨 57~74) |
| 합성 pool | `task2_synthesized/images/syn_*.png` + `annotations/_annotations.coco.json` |
| 모델/학습 | RF-DETR **medium**, 100 epochs, early stopping **없음** — Task 0/1/2 공통 고정 |
| 분할 | StratifiedGroupKFold 5-fold (희소 클래스 층화, 구성 코드 그룹핑) |
| test 추론 | 5-fold 앙상블 → **WBF** 융합 → score ≥ 0.3 제출 |
| 제출 파일 | `submission_task2_medium_task2_syn74.csv` |

**파이프라인 (셀 순서)**
1. 패키지 설치 → 2. 팀 저장소 clone + 함수 import → 3. Kaggle Input 경로 자동 탐색 → 4. 실험 설정(config)
5. 원본 train 로드 + corrections 적용 → 6. 데이터 병합: 합성 pool을 원본 category_id 공간으로 되돌려 병합
7. 병합 데이터 클래스별 bbox 시각화 → 8. 카테고리 라벨 매핑 → 9. 5-fold 분할 + fold별 클래스 배분 점검
10. fold 디렉토리 생성 → 11. 5-fold 학습(fold별 학습곡선/오답 시각화 자동) → 12. 결과 요약(fold 평균·클래스별 mAP)
13. test 5-fold 앙상블 추론 → 14. WBF 융합 → 15. 클래스별 예측 시각화(confidence 표시) → 16. 제출 CSV 생성

**실행 전제**
- Kaggle Notebook: **GPU on / Internet on**
- Input 연결: competition 데이터(`train_images`, `train_annotations`, `test_images`) + 합성 pool 데이터셋(`task2_synthesized`)
- ⚠ medium × 100ep × 5fold는 세션 한도(12h)를 넘길 수 있음. `train_fold()`는 `backup_dir`에 체크포인트가 이미 있으면 그 fold를 건너뛰므로, 이전 커밋 output을 Input으로 추가해 복사하면 **이어하기** 가능 (11번 앞 선택 셀 참고).
- 원칙: **저장소에 등록된 파일/함수는 수정하지 않는다.** 그대로 쓰기 어려운 부분만 노트북 내 로컬 함수로 재정의해 우회한다.


In [ ]:
# [1. 환경 준비] 필요 패키지 설치
#  - rfdetr[train]  : RF-DETR 학습/추론 (metrics.csv 로깅 포함)
#  - ensemble-boxes : 5-fold 예측 WBF(Weighted Box Fusion) 융합용
#  - torchmetrics   : mAP 계산 (저장소 visualize.py가 사용)
!pip install -q "rfdetr[train]" ensemble-boxes torchmetrics

In [ ]:
# [2. 저장소 준비] 팀 저장소 main 브랜치를 clone하고 RF_DETR_split_ver를 import 경로에 추가합니다.
#  ⚠ 저장소 파일/함수는 수정하지 않고 그대로 재사용합니다. (수정이 필요한 로직은 노트북 내 로컬 함수로 재정의)
import os, sys, re, glob, json, shutil
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

REPO_URL = 'https://github.com/kim-tae-yoon-0718/ai12-team01.git'
REPO_DIR = '/kaggle/working/ai12-team01'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
sys.path.insert(0, os.path.join(REPO_DIR, 'RF_DETR_split_ver'))

# ── 저장소에서 그대로 재사용하는 함수 목록 ─────────────────────────────
from dataset import (
    load_raw_annotations,     # 박스당 1개로 흩어진 원본 annotation을 파일명 기준 병합 로드
    apply_corrections,        # corrections.json 기반 bbox 좌표/중복/누락/클래스 오기재 수정
    build_category_mapping,   # category_id 오름차순 -> 라벨 1~N 매핑
    make_folds,               # StratifiedGroupKFold 5-fold 분할
    cache_images,             # 이미지 로컬 캐시 (폴더 하나당 1회 호출)
    write_fold_dirs,          # fold별 {train,valid}에 COCO json + 이미지 배치
    save_label_map,           # label_map.json 저장 (run_kfold가 읽어감)
    compute_label_counts,     # 클래스(라벨)별 박스 수 집계
)
from model import get_rfdetr_model          # RF-DETR variant 생성 (checkpoint_path로 가중치 로드)
from train import run_kfold                 # fold 학습 + fold별 리포팅(mAP/오답 시각화) 루프
from utils import (
    summarize_kfold_results,  # fold 평균 mAP 요약 출력
    summarize_per_class,      # 클래스별 mAP 집계 DataFrame
    show_error_gallery,       # 저장된 PNG 폴더를 grid로 보여주는 범용 뷰어
)
from visualize import collect_predictions_ensemble   # test 폴더에 대한 fold 모델 합집합 추론
from ensemble_boxes import weighted_boxes_fusion     # WBF


In [ ]:
# [corrections 하드코딩] 저장소의 corrections.json을 파일에서 읽지 않고 노트북에 직접 정의합니다.
#  apply_corrections()는 "파일 경로"를 받는 시그니처라(저장소 함수 무수정 원칙),
#  dict를 /kaggle/working에 json으로 1회 저장한 뒤 그 경로를 넘기는 방식으로 우회합니다.
#  적용 순서(coord_fix -> remove -> modify -> add -> fix_category)는 함수 내부에서 보장됩니다.
corrections = {
    # 1) 좌표 오염 수정: 동일 bbox를 가진 항목을 corrected로 교체
    "coord_fix": {
        "K-003351-016262-018357_0_2_0_2_75_000_200.png": [
            {"original": [6567, 625, 311, 315], "corrected": [567, 625, 311, 315]}
        ]
    },
    # 2) 중복/오류 박스 제거: category_id + bbox가 일치하는 첫 항목 1개 제거
    "remove_boxes": {
        "K-001900-016548-019607-033009_0_2_0_2_70_000_200.png": [
            {"category_id": 16548, "bbox": [88, 255, 366, 209]}
        ]
    },
    # 3) 좌표 수정: category_id(+match_bbox) 첫 매치의 bbox를 교체하거나 directive 적용
    "modify_boxes": {
        "K-003351-020014-020238_0_2_0_2_90_000_200.png": [
            {"category_id": 3351, "match_bbox": None, "new_bbox": [390, 260, 170, 165]}
        ],
        "K-003351-019232-029667_0_2_1_2_70_000_200.png": [
            {"category_id": 19232, "match_bbox": None, "directive": "EXTEND_DOWN_95"}
        ]
    },
    # 4) 누락 박스 추가 (원본에 없던 박스라 annotation_id는 None으로 채워짐)
    "add_boxes": {
        "K-001900-016548-019607-033009_0_2_0_2_70_000_200.png": [
            {"category_id": 16548, "bbox": [90, 870, 245, 240]}
        ],
        "K-003351-013900-021325_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [400, 830, 180, 180]}
        ],
        "K-003351-013900-036637_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [440, 880, 175, 175]}
        ],
        "K-003351-020014-022074_0_2_0_2_90_000_200.png": [
            {"category_id": 20014, "bbox": [65, 720, 325, 315]}
        ],
        "K-003351-020238-031863_0_2_0_2_70_000_200.png": [
            {"category_id": 20238, "bbox": [580, 290, 215, 215]}
        ],
        "K-003351-021325-032310_0_2_0_2_90_000_200.png": [
            {"category_id": 32310, "bbox": [595, 830, 345, 245]}
        ],
        "K-003351-029667-031863_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [375, 870, 165, 165]}
        ],
        "K-003351-032310-038162_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [390, 855, 185, 185]}
        ],
        "K-003351-033880-038162_0_2_0_2_75_000_200.png": [
            {"category_id": 33880, "bbox": [70, 600, 310, 425]}
        ],
        "K-003351-035206-041768_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [460, 875, 180, 180]}
        ],
        "K-003544-004543-012247-016548_0_2_0_2_90_000_200.png": [
            {"category_id": 4543, "bbox": [640, 195, 205, 190]}
        ]
    },
    # 5) category_id 오기재 수정: 원본 annotation_id -> 올바른 category_id
    #    (키는 문자열이어야 함 - apply_corrections 내부에서 int(k)로 변환)
    "fix_category": {
        "791": 31863,
        "3444": 3351,
        "3441": 3351,
        "1420": 35206,
        "1412": 27733
        }
    }

CORRECTIONS_PATH = '/kaggle/working/corrections.json'
with open(CORRECTIONS_PATH, 'w', encoding='utf-8') as f:
    json.dump(corrections, f, ensure_ascii=False, indent=2)
    
def count_items(v):
    if isinstance(v, dict):
        try:
            return sum(len(i) if hasattr(i, '__len__') else 1 for i in v.values())
        except:
            return len(v)
    elif hasattr(v, '__len__'):
        return len(v)
    else:
        return 1

print('corrections 하드코딩 저장:', CORRECTIONS_PATH,
      '| 항목:', {k: count_items(corrections[k]) for k in corrections})

In [ ]:
# [3. 입력 경로 자동 탐색]
#  /kaggle/input 아래에서 폴더 "이름"으로 찾기 때문에 competition/데이터셋 슬러그를 몰라도 동작합니다.
#  탐색 실패 시 None이 출력되므로, 그 경우 아래 상수에 실제 경로를 직접 채워 넣으세요.
def find_input_dir(name):
    """/kaggle/input 아래에서 이름이 name인 디렉토리를 찾아 반환합니다 (여러 개면 첫 번째)."""
    hits = sorted(p for p in glob.glob(os.path.join('/kaggle/input', '**', name), recursive=True)
                  if os.path.isdir(p))
    if len(hits) > 1:
        print(f"'{name}' 후보 {len(hits)}개 -> 첫 번째 사용:\n  " + "\n  ".join(hits))
    return hits[0] if hits else None

TRAIN_IMG_DIR = find_input_dir('train_images')       # 원본 train 이미지 (png)
TRAIN_ANN_DIR = find_input_dir('train_annotations')  # 원본 annotation (박스당 json 1개)
TEST_IMG_DIR  = find_input_dir('test_images')        # 제출 대상 test 이미지

POOL_DIR      = find_input_dir('task2_synthesized')   # 합성 pool 루트 (images/, annotations/)
POOL_IMG_DIR  = os.path.join(POOL_DIR, 'images') if POOL_DIR else None
POOL_ANN_PATH = ((glob.glob(os.path.join(POOL_DIR, 'annotations', '*.json')) or [None])[0]
                 if POOL_DIR else None)

paths = {'TRAIN_IMG_DIR': TRAIN_IMG_DIR, 'TRAIN_ANN_DIR': TRAIN_ANN_DIR, 'TEST_IMG_DIR': TEST_IMG_DIR,
         'POOL_IMG_DIR': POOL_IMG_DIR, 'POOL_ANN_PATH': POOL_ANN_PATH}
for k, v in paths.items():
    print(f'{k:14s}:', v)
assert all(paths.values()), "자동 탐색에 실패한 경로가 있습니다. 위 상수에 직접 경로를 지정하세요."

In [ ]:
# [4. 실험 설정]
#  Task 0/1/2 비교 실험이므로 학습 하이퍼파라미터는 세 노트북 모두 아래와 동일하게 고정합니다.
#   - RF-DETR medium / 100 epochs / early stopping 미사용(끝까지 학습) / cosine LR
#   - batch_size 4 x grad_accum_steps 4 = 유효 배치 16 (Kaggle 16GB GPU 기준, OOM 시 2x8로 조정)
#   - 저장소 config.yaml(Colab 경로 기준)은 수정하지 않고, 같은 "스키마"의 dict를 노트북에서 직접 정의합니다.
#     (train.run_kfold / train_fold가 이 스키마를 그대로 소비하므로 저장소 함수와 완전 호환)
TASK_ID = 2

config = {
    'data': {
        'dataset_dir': '/kaggle/tmp/dataset',    # fold 디렉토리 루트 (세션 임시 영역 - output 용량 절약)
        'cache_dir':   '/kaggle/tmp/img_cache',  # 이미지 로컬 캐시
        'n_splits': 5,
        'seed': 42,
    },
    'model': {
        'variant': 'medium',
        'tag': 'medium_task2_syn74',                        # 체크포인트/그래프/제출 파일명에 사용되는 실험 태그
    },
    'train': {
        'epochs': 100,
        'batch_size': 4,
        'grad_accum_steps': 4,
        'lr': 1e-4,
        'lr_encoder': 1.5e-4,
        'weight_decay': 1e-4,
        'lr_scheduler': 'cosine',
        'warmup_epochs': 0.0,
        'lr_min_factor': 0.0,
        'early_stopping': False,                 # Task 간 조건 통일: 100 epoch 전부 학습
        'early_stopping_patience': 10,           # early_stopping=False라 미사용이지만 train_fold 시그니처상 필요
        'early_stopping_min_delta': 0.001,
        'tensorboard': False,
    },
    'output': {
        'local_output_dir': '/kaggle/tmp/outputs',   # fold 학습 중 임시 산출물
        'backup_dir': '/kaggle/working/outputs',     # best 체크포인트/학습곡선/오답 이미지 (커밋 시 보존됨)
    },
}

COLLECT_SCORE_THR = 0.05   # test 추론 "수집" threshold (WBF 전 단계이므로 낮게 잡아 recall 확보)
WBF_IOU_THR       = 0.55   # WBF에서 같은 객체로 간주할 IoU 기준
SUBMIT_SCORE_THR  = 0.3    # 제출 CSV에 포함할 최소 confidence (mAP 채점 가정)

for d in (config['data']['cache_dir'], config['data']['dataset_dir'],
          config['output']['local_output_dir'], config['output']['backup_dir']):
    os.makedirs(d, exist_ok=True)

SUBMISSION_PATH = f"/kaggle/working/submission_task{TASK_ID}_{config['model']['tag']}.csv"
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU(!) - 가속기 설정 확인')
print('제출 파일 경로:', SUBMISSION_PATH)

In [ ]:
# [5. 원본 train 로드 + annotation 수정]
#  원본은 "박스 1개당 JSON 1개"로 흩어져 있어 load_raw_annotations()가 파일명 기준으로 병합합니다.
#  이어서 corrections.json(팀에서 검수한 좌표 오류/중복/누락/클래스 오기재 내역)을 적용합니다.
#  (apply_corrections는 coord_fix -> remove -> modify -> add -> fix_category 순서를 내부에서 보장)
boxes_by_image, cats_by_image, img_meta, ids_by_image = load_raw_annotations(TRAIN_ANN_DIR)
print(f"수정 전: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

boxes_by_image, cats_by_image = apply_corrections(
    boxes_by_image, cats_by_image, ids_by_image, CORRECTIONS_PATH)
print(f"수정 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

# 원본 train의 클래스 목록(56종)을 보존해 둡니다.
#  - Task1: pool이 이 범위를 벗어나지 않는지 검증에 사용
#  - Task2: "train 56종 -> 라벨 1~56" 매핑 검증에 사용
train_cats = sorted({c for cs in cats_by_image.values() for c in cs})
print('train 클래스 수:', len(train_cats))

In [ ]:
# [6. 합성 pool 병합] pool2 (test 전용 18종 + train 희소 클래스 균형 배치)
#  pool의 _annotations.coco.json은 "라벨 네임스페이스"(id 1~N, name=원본 category_id)로 작성되어 있습니다.
#  train 쪽 파이프라인 함수들(build_coco, make_folds 등)은 전부 "원본 category_id" 기준으로 동작하므로,
#  categories의 name 필드를 이용해 원본 id 공간으로 되돌린 뒤 병합합니다. (로컬 함수로 우회)
#  annotation id도 함께 수집합니다 (시각화에서 ann_id 표시용. pool JSON 자체의 id라서
#  train의 원본 annotation id와 번호가 겹칠 수 있으나, 표시 용도로만 쓰므로 무방).
def load_pool_annotations(pool_ann_path):
    """합성 pool COCO json -> (boxes, cats(원본 id 공간), ids, img_meta, 원본 coco dict)"""
    with open(pool_ann_path, encoding='utf-8') as f:
        coco = json.load(f)
    # 라벨 -> 원본 category_id (name 필드가 원본 id 문자열, id 0은 RF-DETR용 더미 'pill')
    label2cat_pool = {c['id']: int(c['name']) for c in coco['categories'] if c['id'] != 0}
    fn_by_img_id = {im['id']: im['file_name'] for im in coco['images']}
    p_boxes, p_cats, p_ids, p_meta = defaultdict(list), defaultdict(list), defaultdict(list), {}
    for im in coco['images']:
        p_meta[im['file_name']] = (im['width'], im['height'])
    for a in coco['annotations']:
        fn = fn_by_img_id[a['image_id']]
        p_boxes[fn].append([float(v) for v in a['bbox']])
        p_cats[fn].append(label2cat_pool[a['category_id']])
        p_ids[fn].append(a.get('id'))
    return p_boxes, p_cats, p_ids, p_meta, coco

pool_boxes, pool_cats, pool_ids, pool_meta, pool_coco = load_pool_annotations(POOL_ANN_PATH)
print(f"pool: 이미지 {len(pool_meta)}장 / 박스 {sum(len(v) for v in pool_boxes.values())}개"
      f" / 클래스 {len({c for cs in pool_cats.values() for c in cs})}종")

# 안전장치 1: 파일명 충돌 확인 (syn_*.png vs K-*.png 라 충돌 없어야 정상)
overlap = set(pool_meta) & set(boxes_by_image)
assert not overlap, f'train/pool 파일명 충돌: {sorted(overlap)[:5]}'

# 안전장치 2: 박스가 하나도 없는 pool 이미지는 학습에 기여하지 못하고
#             make_folds의 층화 기준(이미지 내 최희소 클래스) 계산에서 에러가 나므로 제외
empty_pool = [fn for fn in pool_meta if not pool_boxes.get(fn)]
if empty_pool:
    print(f'박스 0개인 pool 이미지 {len(empty_pool)}장 제외:', empty_pool[:5])

for fn in pool_meta:
    if pool_boxes.get(fn):
        boxes_by_image[fn] = pool_boxes[fn]
        cats_by_image[fn] = pool_cats[fn]
        ids_by_image[fn] = pool_ids[fn]   # 시각화(ann_id 표시)에서 사용
        img_meta[fn] = pool_meta[fn]

print(f"병합 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

In [ ]:
# [6-1. 데이터 제외] 7번 셀 시각화에서 발견한 불량 이미지/박스를 학습 대상에서 제외합니다.
#  사용법: 7번 셀 plot 제목의 파일명과 ann_id를 보고 아래 목록을 채운 뒤,
#          이 셀부터 아래(7번 시각화 -> 8번 매핑 -> 9번 fold...)를 다시 실행하세요.
#  ⚠ 반드시 8번(매핑)/9번(fold 분할) "이전"에 실행되어야 합니다.

# (1) 이미지 통째로 제외: 파일명 나열 (예: 배치가 잘못된 합성 이미지)
EXCLUDE_FILES = [
    'syn_00505.png',
    'syn_00102.png'
]


for fn in EXCLUDE_FILES:
    if fn in boxes_by_image:
        for d in (boxes_by_image, cats_by_image, ids_by_image, img_meta):
            d.pop(fn, None)
        print('이미지 제외:', fn)
    else:
        print('⚠ 목록에 없는 파일명(오타 확인):', fn)


print(f"제외 처리 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

In [ ]:
# [7. 병합 데이터 시각화] 클래스별로 GT bbox "전부"를 잘라(grid) 모아봅니다.
#  저장소에는 "예측" crop 유틸(crop_predictions_by_class)만 있고 GT bbox crop용 함수는 없어 로컬로 정의합니다.
#  합성 이미지(syn_*.png)의 bbox도 함께 표시되므로 원본과 섞였을 때의 품질(크기/배경/겹침)을 함께 점검하세요.
#  ⚠ Kaggle 기본 matplotlib에는 한글 폰트가 없으므로, plot에 렌더링되는 텍스트는 영어로만 표기합니다.
#  각 crop 제목: 1줄째 = 파일명(전체), 2줄째 = annotation_id
#   - train 박스: 원본 annotation JSON의 id (corrections의 add_boxes로 추가된 박스는 None)
#   - pool 박스: pool JSON의 id (6번 셀에서 수집)
import math

IMG_PATH_INDEX = {os.path.basename(p): p
                  for p in glob.glob(os.path.join(TRAIN_IMG_DIR, '**', '*.png'), recursive=True)}
IMG_PATH_INDEX.update({os.path.basename(p): p
                       for p in glob.glob(os.path.join(POOL_IMG_DIR, '**', '*.png'), recursive=True)})
print('이미지 경로 인덱스:', len(IMG_PATH_INDEX), '개 파일')

def show_gt_class_crops(boxes_by_image, cats_by_image, ids_by_image, img_path_index,
                        ncols=6, pad=8, classes=None):
    """클래스(원본 category_id)별 GT bbox crop을 '전부' grid로 표시합니다.

    Args:
        boxes_by_image / cats_by_image: 병합 완료된 annotation dict (COCO xywh, 원본 id)
        ids_by_image (dict): file_name -> [annotation_id, ...] (boxes와 순서 동기화 상태)
        img_path_index (dict): file_name -> 이미지 실제 경로
        ncols (int): 한 줄에 표시할 crop 수 (행 수는 박스 수에 따라 자동 결정)
        pad (int): crop 시 bbox 주변 여백(px)
        classes (list): 지정하면 해당 category_id들만 표시 (None이면 전체)
    """
    by_cat = defaultdict(list)   # category_id -> [(file_name, bbox, ann_id), ...]
    for fn, cats in cats_by_image.items():
        ids = ids_by_image.get(fn)
        if not ids or len(ids) != len(boxes_by_image[fn]):   # 동기화가 깨졌으면 표시만 생략 (방어적)
            ids = [None] * len(boxes_by_image[fn])
        for c, b, aid in zip(cats, boxes_by_image[fn], ids):
            by_cat[c].append((fn, b, aid))

    targets = sorted(by_cat) if classes is None else [c for c in classes if c in by_cat]
    for c in targets:
        items = by_cat[c]
        # 같은 이미지를 bbox마다 다시 읽지 않도록, 클래스 단위로 파일당 1회만 로드
        img_cache = {fn: np.array(Image.open(img_path_index[fn]).convert('RGB'))
                     for fn in {fn for fn, _, _ in items}}

        nrows = math.ceil(len(items) / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(2.2 * ncols, 2.8 * nrows))
        axes = np.atleast_1d(axes).reshape(-1)
        for ax in axes:
            ax.axis('off')
        for ax, (fn, b, aid) in zip(axes, items):
            img = img_cache[fn]
            h, w = img.shape[:2]
            x, y, bw, bh = [int(v) for v in b]
            x1, y1 = max(0, x - pad), max(0, y - pad)
            x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
            ax.imshow(img[y1:y2, x1:x2])
            ax.set_title(f'{fn}\nann_id={aid}', fontsize=5)
            x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
            ax.imshow(img[y1:y2, x1:x2])
            ax.set_title(f'{fn}\nann_id={aid}', fontsize=5)
        fig.suptitle(f'category_id={c}  (total {len(items)} boxes)', fontsize=10)
        plt.tight_layout(rect=[0, 0, 1, 0.97])   # suptitle과 첫 행이 겹치지 않게 상단 여백 확보
        plt.show()

# 전체 클래스의 모든 bbox를 표시합니다. 특정 클래스만 보려면 classes=[3351, 16548] 처럼 지정하세요.
show_gt_class_crops(boxes_by_image, cats_by_image, ids_by_image, IMG_PATH_INDEX)

In [ ]:
# [8. 카테고리 라벨 매핑 - Task 2 전용: 74종]
#  요구 체계: train 56종 -> 라벨 1~56, test 전용 18종 -> 라벨 57~74.
#  pool2 JSON의 categories가 이미 이 체계(1~74, name=원본 category_id)로 작성되어 있으므로
#  이를 신뢰 소스로 사용합니다.
#  ⚠ 저장소 build_category_mapping()을 74종에 그대로 쓰면 "등장 클래스 전체 오름차순"이라
#    test 18종이 1~56 사이에 끼어들어 요구 체계가 깨지므로, 여기서는 로컬 매핑으로 우회합니다.
cat2label = {int(c['name']): c['id'] for c in pool_coco['categories'] if c['id'] != 0}
label2cat = {v: k for k, v in cat2label.items()}
all_cats = [label2cat[l] for l in sorted(label2cat)]   # build_coco()에 넘길 카테고리 목록(라벨 오름차순)

# 검증 1: train 56종이 정확히 1~56에 (오름차순으로) 매핑되는가 -> Task 0/1과 네임스페이스 호환 보장
for i, c in enumerate(sorted(train_cats), start=1):
    assert cat2label.get(c) == i, f'train 클래스 {c}가 라벨 {cat2label.get(c)}에 매핑됨 (기대: {i})'

# 검증 2: 병합 데이터에 등장하는 모든 클래스가 매핑에 존재하는가
merged_cats = {c for cs in cats_by_image.values() for c in cs}
missing = sorted(merged_cats - set(cat2label))
assert not missing, f'매핑에 없는 클래스 존재: {missing}'

test_only_labels = sorted(set(cat2label.values()) - set(range(1, len(train_cats) + 1)))
print(f'전체 {len(cat2label)}종 매핑 | train {len(train_cats)}종 -> 1~{len(train_cats)} 확인'
      f' | test 전용 라벨: {test_only_labels}')

In [ ]:
# [9. 5-fold 분할 + fold별 클래스 배분 점검]
#  make_folds()는 StratifiedGroupKFold를 사용합니다 (저장소 함수 그대로).
#   - group: 파일명의 알약 구성 코드('_0_2' 앞부분) -> 같은 구성이 train/valid에 섞이지 않음
#     합성 이미지(syn_*.png)는 '_0_2' 패턴이 없어 이미지 1장 = 그룹 1개로 취급됩니다.
#   - 층화 기준: 이미지 내 "가장 희소한 클래스" -> 희소 클래스가 fold에 고르게 분산
file_names = sorted(boxes_by_image)
folds = make_folds(file_names, boxes_by_image, cats_by_image, cat2label,
                   config['data']['n_splits'], config['data']['seed'])

def summarize_fold_distribution(folds, file_names, cats_by_image, cat2label):
    """fold별 train/valid 이미지·박스 수와 클래스 커버리지를 점검합니다. (로컬 정의 - 저장소에 없음)

    Returns:
        summary (pd.DataFrame): fold별 요약 (누락 라벨 목록 포함)
        valid_pivot (pd.DataFrame): 라벨 x fold 형태의 valid 박스 수 표
    """
    all_labels = set(cat2label.values())
    rows, val_pivot = [], {}
    for fi, (tr, va) in enumerate(folds):
        def label_box_counts(idxs):
            cnt = defaultdict(int)
            for i in idxs:
                for c in cats_by_image[file_names[i]]:
                    cnt[cat2label[c]] += 1
            return cnt
        tr_cnt, va_cnt = label_box_counts(tr), label_box_counts(va)
        rows.append({
            'fold': fi,
            'train_imgs': len(tr), 'valid_imgs': len(va),
            'train_boxes': sum(tr_cnt.values()), 'valid_boxes': sum(va_cnt.values()),
            'train_missing_labels': sorted(all_labels - set(tr_cnt)),
            'valid_missing_labels': sorted(all_labels - set(va_cnt)),
        })
        val_pivot[f'fold{fi}_valid'] = va_cnt
    summary = pd.DataFrame(rows)
    valid_pivot = pd.DataFrame(val_pivot).reindex(sorted(all_labels)).fillna(0).astype(int)
    valid_pivot.index.name = 'label'
    return summary, valid_pivot

fold_summary, valid_pivot = summarize_fold_distribution(folds, file_names, cats_by_image, cat2label)
display(fold_summary)

# train에서 통째로 빠진 클래스가 있는 fold는 그 클래스를 전혀 학습하지 못하므로 경고로 표시
for _, r in fold_summary.iterrows():
    if r['train_missing_labels']:
        print(f"⚠ fold {r['fold']}: train에 없는 라벨 {r['train_missing_labels']}")
    if r['valid_missing_labels']:
        print(f"(참고) fold {r['fold']}: valid에 없는 라벨 {r['valid_missing_labels']}")

valid_pivot   # 라벨 x fold별 valid 박스 수 (셀 마지막 줄 -> 표로 표시)

In [ ]:
# 이 셀 바로 위에 추가해서 확인
print("=== category mapping 확인 ===")
print(f"총 클래스 수: {len(cat2label)}")
print(f"\n{'category_id':>12} {'dl_idx':>10}")
print("-" * 25)
for name, cat_id in sorted(cat2label.items(), key=lambda x: x[1]):
    print(f"  {cat_id:>10} {name:>10}")

In [ ]:
# [10. fold 디렉토리 생성]
#  이미지를 로컬 캐시에 1회 복사한 뒤, fold별 {train,valid} 폴더에 COCO json + 이미지를 배치합니다.
#  (전부 저장소 함수 - cache_images는 폴더 하나당 1회씩 호출하면 같은 캐시에 누적됩니다)
cache_images(TRAIN_IMG_DIR, config['data']['cache_dir'])

cache_images(POOL_IMG_DIR, config['data']['cache_dir'])    # 합성 이미지도 같은 캐시 폴더에 추가

write_fold_dirs(folds, file_names, boxes_by_image, cats_by_image, img_meta,
                cat2label, all_cats, config['data']['cache_dir'], config['data']['dataset_dir'])

# run_kfold()가 학습 전에 이 label_map.json을 읽으므로 반드시 학습 전에 저장해야 합니다.
save_label_map(cat2label, label2cat, config['data']['dataset_dir'])
print('label_map.json 저장 완료')

## 11. 5-fold 학습

`run_kfold(config)` 하나로 fold마다 다음이 자동 수행됩니다 (모두 저장소 함수):

1. `train_fold()` — 학습 후 best 체크포인트를 `backup_dir`에 백업, **학습곡선 그래프 출력·저장**
2. `report_fold_result()` — valid셋 추론 1회로 **mAP 계산 + 오답 이미지 저장** (`{tag}_fold{i}_errors/`)

**이어하기**: `train_fold()`는 `backup_dir`에 해당 fold의 `*_best.pth`가 이미 있으면 학습을 건너뜁니다.
세션이 끊기면 이전 커밋의 output을 Input으로 추가하고 아래 선택 셀로 복사한 뒤 다시 실행하세요.

⏱ **시간 주의**: medium × 100 epochs × 5 folds는 12h 세션 한도를 넘길 수 있습니다.
처음에는 `run_kfold(config, max_folds=1)`로 fold 1개 소요 시간을 가늠해 보는 것을 권장합니다.

In [ ]:
# (선택) 이어하기: 이전 노트북 커밋의 output(outputs/*.pth)을 Input으로 추가했다면 backup_dir로 복사
#  -> train_fold()가 이미 학습된 fold를 자동으로 건너뜁니다.
# PREV_OUTPUT_DIR = '/kaggle/input/<이전-노트북-output-슬러그>/outputs'
# for p in glob.glob(os.path.join(PREV_OUTPUT_DIR, '*_best.pth')):
#     shutil.copy(p, config['output']['backup_dir'])
#     print('복사:', os.path.basename(p))

In [ ]:
# [11. 5-fold 학습 실행]
#  fold가 끝날 때마다 학습곡선 그래프와 valid mAP, 오답 이미지 저장 경로가 출력됩니다.
#  소요 시간 가늠용 sanity check: run_result = run_kfold(config, max_folds=1)
run_result = run_kfold(config)

In [ ]:
# [12. 결과 요약]
#  - summarize_kfold_results: fold 평균 mAP (mAP@0.75:0.95 평균±표준편차)
#  - summarize_per_class: 클래스별 mAP를 전 fold에 걸쳐 집계 (데이터 내 등장 횟수 포함)
#  (Task2 참고) test 전용 라벨(57~74)의 mAP는 "합성 valid" 기준이므로 실제 test 성능과
#  다를 수 있습니다. 합성-실사 도메인 차이를 감안해 해석하세요.
kfold_summary = summarize_kfold_results(run_result['fold_metrics'], config['model']['tag'])

label_counts = compute_label_counts(config['data']['dataset_dir'])
per_class_df = summarize_per_class(run_result['fold_metrics'],
                                   run_result['label_to_category_id'], label_counts)
per_class_df   # mean_AP 내림차순 - 하위 클래스가 보강 대상/효과 확인 포인트

In [ ]:
# [12-1. 오답 이미지 확인] run_kfold가 fold별로 저장해둔 오답 이미지를 grid로 봅니다.
#  fold 번호/페이지(start, limit)를 바꿔가며 확인하세요. (GT=초록, 예측=빨강, 파일명에 순번 포함)
show_error_gallery(
    os.path.join(config['output']['backup_dir'], f"{config['model']['tag']}_fold0_errors"),
    ncols=3, limit=6,
)

## 13~16. test 추론 → WBF 앙상블 → 클래스별 시각화 → 제출

test에는 GT가 없으므로 mAP 계산 없이:
5개 fold 모델의 예측을 전부 수집(합집합, 저장소 `collect_predictions_ensemble`) → **WBF**로 이미지 단위 융합 →
클래스별 crop 시각화(confidence 표시)로 육안 점검 → 제출 CSV 생성 순서로 진행합니다.

In [ ]:
# [13. test 추론 - 5 fold 합집합 수집]
#  각 fold의 best 체크포인트로 모델 5개를 만들어, test 이미지마다 5개 모델의 예측을 전부 모읍니다.
#  WBF 융합 전이므로 낮은 threshold(COLLECT_SCORE_THR)로 수집하고, 제출 단계에서 최종 필터링합니다.
ckpts = [p for p in run_result['checkpoints'] if p]
assert len(ckpts) == config['data']['n_splits'], f"체크포인트 누락: {run_result['checkpoints']}"

models = [get_rfdetr_model(config['model']['variant'], checkpoint_path=p) for p in ckpts]
test_pred_data = collect_predictions_ensemble(models, TEST_IMG_DIR,
                                              score_threshold=COLLECT_SCORE_THR)
print('test 이미지 수:', len(test_pred_data))

# RF-DETR 내부 예약 라벨(배경 0 등, label_map에 없는 라벨) 제거
#  - 저장소 report_fold_result()가 valid 평가 때 하는 것과 동일한 정제 과정입니다.
valid_labels = set(label2cat)
for d in test_pred_data:
    keep = torch.tensor([int(l) in valid_labels for l in d['pred_labels']], dtype=torch.bool)
    for k in ('pred_boxes', 'pred_labels', 'pred_scores', 'pred_fold'):
        d[k] = d[k][keep]
print('합집합 예측 박스 수:', sum(len(d['pred_boxes']) for d in test_pred_data))

In [ ]:
# [14. WBF(Weighted Box Fusion) 융합]
#  합집합 그대로 제출하면 같은 알약이 fold 수만큼 중복되어 mAP에서 FP로 깎입니다.
#  저장소의 _cluster_same_class_boxes(대표 박스 1개 선택)와 달리, WBF는 겹치는 박스들의 좌표를
#  confidence 가중 평균해 더 정교한 박스를 만듭니다. (그래서 외부 패키지 ensemble-boxes 사용)
#  conf_type='avg'(기본값) 기준, 일부 fold만 잡은 박스는 score가 (동의 모델 수/전체 모델 수) 비율로
#  낮아져 "몇 개 fold가 동의했는지" 신호가 score에 자연스럽게 반영됩니다.
def fuse_predictions_wbf(pred_data, n_models, iou_thr=0.55, skip_box_thr=0.05):
    """collect_predictions_ensemble() 결과를 이미지 단위 WBF로 융합합니다. (로컬 정의)

    Args:
        pred_data: collect_predictions_ensemble()의 반환값 (pred_fold로 모델 구분 가능)
        n_models (int): 앙상블 모델(fold) 수
        iou_thr (float): 같은 객체로 간주할 IoU
        skip_box_thr (float): 융합 전 무시할 최소 score

    Returns:
        list of dicts: [{'file_name', 'image', 'pred_boxes'(xyxy), 'pred_labels', 'pred_scores'}, ...]
    """
    fused = []
    for d in pred_data:
        h, w = d['image'].shape[:2]
        scale = np.array([w, h, w, h], dtype=np.float32)

        if len(d['pred_boxes']) == 0:   # 예측이 하나도 없는 이미지
            fused.append({'file_name': d['file_name'], 'image': d['image'],
                          'pred_boxes': torch.zeros((0, 4), dtype=torch.float32),
                          'pred_labels': torch.zeros((0,), dtype=torch.int64),
                          'pred_scores': torch.zeros((0,), dtype=torch.float32)})
            continue

        # WBF는 모델별 리스트 + 0~1 정규화 좌표를 기대합니다.
        boxes_list, scores_list, labels_list = [], [], []
        for fi in range(n_models):
            m = (d['pred_fold'] == fi).numpy()
            b = d['pred_boxes'].numpy()[m] / scale
            boxes_list.append(np.clip(b, 0.0, 1.0).tolist())
            scores_list.append(d['pred_scores'].numpy()[m].tolist())
            labels_list.append(d['pred_labels'].numpy()[m].tolist())

        boxes, scores, labels = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list,
            iou_thr=iou_thr, skip_box_thr=skip_box_thr)

        fused.append({
            'file_name': d['file_name'],
            'image': d['image'],
            'pred_boxes': torch.tensor(np.asarray(boxes) * scale, dtype=torch.float32),
            'pred_labels': torch.tensor(labels, dtype=torch.int64),
            'pred_scores': torch.tensor(scores, dtype=torch.float32),
        })
    return fused

fused_data = fuse_predictions_wbf(test_pred_data, n_models=len(models),
                                  iou_thr=WBF_IOU_THR, skip_box_thr=COLLECT_SCORE_THR)
n_before = sum(len(d['pred_boxes']) for d in test_pred_data)
n_after = sum(len(d['pred_boxes']) for d in fused_data)
print(f'융합 전 박스 {n_before}개 -> 융합 후 {n_after}개')

In [ ]:
# [15. 추론 결과 클래스별 시각화] WBF 융합 예측을 예측 클래스별 crop grid로 표시합니다.
#  각 crop 제목에 confidence score + 파일명이 표시됩니다.
#  저장소 crop_predictions_by_class()는 fold 정보(pred_fold)를 요구하는데 WBF 이후에는
#  fold 개념이 사라지므로, 융합 결과 전용으로 로컬 정의했습니다.
def show_pred_class_crops(fused_data, label2cat, score_thr=0.3, max_per_class=8, pad=8):
    """WBF 융합 예측을 클래스별 crop grid로 표시합니다 (제목: confidence / 파일명).

    Args:
        fused_data: fuse_predictions_wbf()의 반환값
        label2cat (dict): 모델 라벨 -> 원본 category_id
        score_thr (float): 표시할 예측의 최소 confidence (제출 기준과 동일하게 두고 점검 권장)
        max_per_class (int): 클래스당 표시할 crop 수 (score 내림차순 상위)
        pad (int): crop 여백(px)
    """
    by_label = defaultdict(list)
    for d in fused_data:
        h, w = d['image'].shape[:2]
        keep = d['pred_scores'] >= score_thr
        for box, label, score in zip(d['pred_boxes'][keep], d['pred_labels'][keep],
                                     d['pred_scores'][keep]):
            x1, y1, x2, y2 = box.tolist()
            x1, y1 = max(0, int(x1) - pad), max(0, int(y1) - pad)
            x2, y2 = min(w, int(x2) + pad), min(h, int(y2) + pad)
            if x2 <= x1 or y2 <= y1:
                continue
            by_label[int(label)].append((d['image'][y1:y2, x1:x2], float(score), d['file_name']))

    print(f'score >= {score_thr} 기준, 예측이 존재하는 클래스: {len(by_label)}개')
    for label in sorted(by_label):
        items = sorted(by_label[label], key=lambda t: -t[1])[:max_per_class]
        fig, axes = plt.subplots(1, max_per_class, figsize=(2.2 * max_per_class, 2.6))
        axes = np.atleast_1d(axes)
        for ax in axes:
            ax.axis('off')
        for ax, (crop, score, fn) in zip(axes, items):
            ax.imshow(crop)
            ax.set_title(f'{score:.2f}\n{fn[:16]}', fontsize=6)
        fig.suptitle(f'label={label} / category_id={label2cat[label]}'
                     f'  (score>={score_thr}: {len(by_label[label])}건)', fontsize=9, y=1.05)
        plt.tight_layout()
        plt.show()

show_pred_class_crops(fused_data, label2cat, score_thr=SUBMIT_SCORE_THR, max_per_class=8)

In [ ]:
# [16. 제출 CSV 생성]
#  요구 포맷: annotation_id, image_id, category_id, bbox_x, bbox_y, bbox_w, bbox_h, score
#   - image_id: 이미지 "파일명의 숫자"
#   - category_id: 원본 category_id (모델 라벨 -> label2cat 역매핑)
#   - annotation_id: 행마다 고유한 임의 값 (여기서는 1부터 증가)
#   - bbox: xyxy(내부 표현) -> xywh(COCO)로 변환
def extract_image_id(file_name):
    """파일명에서 숫자를 추출해 image_id로 사용합니다.
    숫자 블록이 2개 이상이면 어떤 규칙인지 판단할 수 없으므로 일부러 에러를 내어 확인을 요구합니다.
    (그 경우 test 파일명 규칙에 맞게 이 함수만 수정하면 됩니다)
    """
    stem = os.path.splitext(file_name)[0]
    digits = re.findall(r'\d+', stem)
    assert len(digits) == 1, f'파일명 숫자 규칙 확인 필요: {file_name} -> {digits}'
    return int(digits[0])

def make_submission(fused_data, label2cat, score_thr, out_path):
    """WBF 융합 예측을 제출 포맷 DataFrame으로 만들어 CSV 저장합니다. (로컬 정의)"""
    rows, ann_id, n_empty = [], 1, 0
    for d in fused_data:
        image_id = extract_image_id(d['file_name'])
        keep = d['pred_scores'] >= score_thr
        if int(keep.sum()) == 0:
            n_empty += 1
        for box, label, score in zip(d['pred_boxes'][keep], d['pred_labels'][keep],
                                     d['pred_scores'][keep]):
            x1, y1, x2, y2 = box.tolist()
            rows.append({
                'annotation_id': ann_id,
                'image_id': image_id,
                'category_id': label2cat[int(label)],
                'bbox_x': round(x1, 2), 'bbox_y': round(y1, 2),
                'bbox_w': round(x2 - x1, 2), 'bbox_h': round(y2 - y1, 2),
                'score': round(float(score), 4),
            })
            ann_id += 1
    df = pd.DataFrame(rows, columns=['annotation_id', 'image_id', 'category_id',
                                     'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'score'])
    df.to_csv(out_path, index=False)
    print(f'저장 완료: {out_path}')
    print(f'총 {len(df)}행 / 이미지 {len(fused_data)}장 (예측 0건 이미지: {n_empty}장)')
    if n_empty:
        print('⚠ 예측이 하나도 없는 이미지가 있습니다. score_thr를 낮추거나 해당 이미지를 육안 확인하세요.')
    return df

# 파일명 -> image_id 매핑을 눈으로 먼저 확인 (규칙이 다르면 extract_image_id 수정)
for d in fused_data[:5]:
    print(d['file_name'], '->', extract_image_id(d['file_name']))

submission_df = make_submission(fused_data, label2cat, SUBMIT_SCORE_THR, SUBMISSION_PATH)
submission_df.head(10)